# Gun 1 - LLM ile Otomatik Etiket/Sinif Cikarimi (DOC-24)

OCR ile cikarilan belge metnini bir LLM'e (Claude) vererek belgeyi onceden tanimlanmis bir sinifa (fatura/sozlesme/dilekce/talep formu vb.) atiyor ve serbest metinli etiketler cikariyoruz. `src/classifier.py` modulunu iki asamada test ediyoruz:

1. **Sahte (mock) istemciyle yerel mantik testi** — API anahtari gerektirmez, JSON ayristirma ve fallback davranisini dogrular.
2. **Gercek API ile uctan uca test** — `data/processed/ocr_real_outputs.json` icindeki 5 gercek belge uzerinde calisir; sonuclar `data/processed/classification_report.json` dosyasina kaydedilir.

In [1]:
import sys
import os
import json

sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))
from classifier import classify_document, classify_chunks, DEFAULT_CATEGORIES

print("classifier modulu yuklendi.")
print(f"Varsayilan siniflar: {DEFAULT_CATEGORIES}")

classifier modulu yuklendi.
Varsayilan siniflar: ['fatura', 'sözleşme', 'dilekçe', 'talep formu', 'diğer']


## 1. Sahte istemciyle yerel mantik testi

In [2]:
class FakeBlock:
    def __init__(self, text):
        self.type = "text"
        self.text = text


class FakeResponse:
    def __init__(self, text):
        self.content = [FakeBlock(text)]


class FakeMessages:
    def __init__(self, reply):
        self.reply = reply

    def create(self, **kwargs):
        return FakeResponse(self.reply)


class FakeClient:
    def __init__(self, reply):
        self.messages = FakeMessages(reply)

In [3]:
mock_reply = json.dumps({
    "sinif": "talep formu",
    "guven": 0.92,
    "etiketler": ["donanim", "monitor", "ic talep"],
    "gerekce": "Belge bir calisanin ekipman talebi icin yazdigi ic yazi.",
}, ensure_ascii=False)

result = classify_document("Talep Eden: Dila Alpay...", client=FakeClient(mock_reply))
assert result["sinif"] == "talep formu"
print("OK - normal JSON yaniti dogru ayristirildi:")
print(json.dumps(result, ensure_ascii=False, indent=2))

OK - normal JSON yaniti dogru ayristirildi:
{
  "sinif": "talep formu",
  "guven": 0.92,
  "etiketler": [
    "donanim",
    "monitor",
    "ic talep"
  ],
  "gerekce": "Belge bir calisanin ekipman talebi icin yazdigi ic yazi."
}


In [4]:
fenced_reply = "```json\n" + mock_reply + "\n```"
result_fenced = classify_document("metin", client=FakeClient(fenced_reply))
assert result_fenced["sinif"] == "talep formu"
print("OK - markdown kod blogu icindeki JSON de dogru ayristirildi.")

OK - markdown kod blogu icindeki JSON de dogru ayristirildi.


In [5]:
bad_reply = json.dumps({"sinif": "bilinmeyen_sinif", "guven": 0.5, "etiketler": [], "gerekce": "x"})
result_bad = classify_document("metin", client=FakeClient(bad_reply))
assert result_bad["sinif"] == "diğer"
print(f"OK - listede olmayan sinif otomatik olarak fallback'e dustu: '{result_bad['sinif']}'")

OK - listede olmayan sinif otomatik olarak fallback'e dustu: 'diğer'


In [6]:
try:
    classify_document("   ")
    print("FAIL - bos text icin ValueError beklenirdi")
except ValueError:
    print("OK - bos text icin ValueError firlatildi.")

OK - bos text icin ValueError firlatildi.


## 2. Gercek belge verisiyle uctan uca test

In [7]:
OCR_PATH = "../data/processed/ocr_real_outputs.json"
with open(OCR_PATH, encoding="utf-8") as f:
    ocr_outputs = json.load(f)


def format_document(fields: dict) -> str:
    return (
        f"Talep Eden: {fields['talep_eden']}\n"
        f"Tarih: {fields['tarih']}\n"
        f"Departman: {fields['departman']}\n"
        f"Konu: {fields['konu']}\n\n"
        f"{fields['aciklama']}"
    )


print(f"{len(ocr_outputs)} belge yuklendi: {list(ocr_outputs.keys())}")

5 belge yuklendi: ['test_talep_01.png', 'test_talep_02.png', 'test_talep_03.png', 'test_talep_04.png', 'test_talep_05.png']


In [8]:
classification_report = {}

for filename, fields in sorted(ocr_outputs.items()):
    document_text = format_document(fields)
    try:
        result = classify_document(document_text)
        classification_report[filename] = result
        print(f"{filename}: sinif={result['sinif']!r}, guven={result.get('guven')}, etiketler={result.get('etiketler')}")
    except Exception as e:
        classification_report[filename] = {"hata": str(e)}
        print(f"{filename}: HATA - {e}")

test_talep_01.png: sinif='talep formu', guven=0.95, etiketler=['ek monitör talebi', 'yazılım geliştirme', 'donanım talebi', 'iş talebi']


test_talep_02.png: sinif='talep formu', guven=0.9, etiketler=['laptop talebi', 'insan kaynaklari', 'ekipman talebi', 'iş verimliliği']


test_talep_03.png: sinif='talep formu', guven=0.9, etiketler=['klavye değişimi', 'arıza bildirimi', 'ekipman talebi', 'talep eden: zeynep kaya']


test_talep_04.png: sinif='talep formu', guven=0.9, etiketler=['yazıcı talebi', 'muhasebe departmanı', 'arıza bildirimi', 'ekipman talebi']


test_talep_05.png: sinif='talep formu', guven=0.9, etiketler=['ek ekran talebi', 'pazarlama departmani', 'donanım talebi', 'verimlilik']


In [9]:
OUT_PATH = "../data/processed/classification_report.json"

if any("hata" not in v for v in classification_report.values()):
    with open(OUT_PATH, "w", encoding="utf-8") as f:
        json.dump(classification_report, f, ensure_ascii=False, indent=2)
    print(f"Sonuclar '{OUT_PATH}' dosyasina kaydedildi.")
else:
    print(
        "Gercek API cagrisi basarisiz oldu (muhtemelen ANTHROPIC_API_KEY "
        ".env dosyasinda tanimli degil). Sonuc dosyasi guncellenmedi; "
        ".env dosyasina gecerli bir anahtar eklenip bu hucre yeniden "
        "calistirilabilir."
    )

Sonuclar '../data/processed/classification_report.json' dosyasina kaydedildi.
